# Spherical HMI Extrapolation

This notebook runs a compact spherical NF2 example from HMI synoptic data. It downloads an input magnetogram, trains the spherical model, exports a volume, and plots radial-field diagnostics.

## 1. Configure The Spherical Run

Set the Carrington rotation, output paths, sampling resolution, and radial domain. The defaults are intentionally modest for a release notebook.

In [ ]:
from pathlib import Path

RUN_JSOC_DOWNLOAD = True
RUN_TRAINING = True

JSOC_EMAIL = "you@example.org"
HMI_TIME = "2024.05.10_00:00:00_TAI"
CARRINGTON_ROTATION = 2283
FULL_DISK_SERIES = "B_720s"
SYNOPTIC_SERIES = "b_synoptic"
SYNOPTIC_SEGMENTS = "Br,Bt,Bp"

RUN_DIR = Path("runs/spherical_hmi")
DATA_DIR = RUN_DIR / "data"
WORK_DIR = RUN_DIR / "work"
NF2_PATH = RUN_DIR / "extrapolation_result.nf2"
EXPORT_DIR = RUN_DIR / "exports"

FULL_DISK_BR = DATA_DIR / "full_disk.Br.fits"
FULL_DISK_BT = DATA_DIR / "full_disk.Bt.fits"
FULL_DISK_BP = DATA_DIR / "full_disk.Bp.fits"
FULL_DISK_BR_ERR = FULL_DISK_BR
FULL_DISK_BT_ERR = FULL_DISK_BT
FULL_DISK_BP_ERR = FULL_DISK_BP
SYNOPTIC_BR = DATA_DIR / "synoptic.Br.fits"
SYNOPTIC_BT = DATA_DIR / "synoptic.Bt.fits"
SYNOPTIC_BP = DATA_DIR / "synoptic.Bp.fits"

SPHERICAL_SAMPLING = [24, 48, 96]
RADIUS_RANGE_SOLRAD = [1.0, 1.2]

## 2. Import Packages

W&B console capture is disabled for notebook stability while keeping metric and image logging active.

In [ ]:
import os
os.environ.setdefault("WANDB_CONSOLE", "off")

from pathlib import Path
from dateutil.parser import parse

from astropy import units as u
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
import wandb

import nf2
from nf2.evaluation.metric import normalized_divergence, sigma_J, theta_J, vector_norm

## 3. Download The HMI Map

Download the requested HMI synoptic map and keep the local path available for both training and later reference plots.

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)

def first_match(directory, pattern):
    matches = sorted(Path(directory).glob(pattern))
    if not matches:
        raise FileNotFoundError(f"No files matching {pattern!r} found in {directory}.")
    return matches[0]

if RUN_JSOC_DOWNLOAD:
    hmi_time = parse(HMI_TIME.replace("_TAI", "").replace(".", "-", 2).replace("_", "T"))
    full_disk_dir = DATA_DIR / "full_disk"
    synoptic_dir = DATA_DIR / "synoptic"

    nf2.download_hmi_full_disk(str(full_disk_dir), JSOC_EMAIL, hmi_time, series=FULL_DISK_SERIES)
    nf2.download_hmi_synoptic(str(synoptic_dir), JSOC_EMAIL, carrington_rotation=CARRINGTON_ROTATION, segments=SYNOPTIC_SEGMENTS, series=SYNOPTIC_SERIES)

    FULL_DISK_BR = first_match(full_disk_dir, "*Br.fits")
    FULL_DISK_BT = first_match(full_disk_dir, "*Bt.fits")
    FULL_DISK_BP = first_match(full_disk_dir, "*Bp.fits")
    SYNOPTIC_BR = first_match(synoptic_dir, "*Br.fits")
    SYNOPTIC_BT = first_match(synoptic_dir, "*Bt.fits")
    SYNOPTIC_BP = first_match(synoptic_dir, "*Bp.fits")
    FULL_DISK_BR_ERR, FULL_DISK_BT_ERR, FULL_DISK_BP_ERR = FULL_DISK_BR, FULL_DISK_BT, FULL_DISK_BP

print("Full-disk files:")
print(FULL_DISK_BR)
print(FULL_DISK_BT)
print(FULL_DISK_BP)
print("Synoptic files:")
print(SYNOPTIC_BR)
print(SYNOPTIC_BT)
print(SYNOPTIC_BP)

## 4. Build And Train The Configuration

The spherical configuration defines the map boundary, random spherical sampling volume, validation slices, and training callbacks.

In [ ]:
config = {
    "path": str(RUN_DIR),
    "work_path": str(WORK_DIR),
    "logging": {"project": "nf2", "name": "Spherical HMI"},
    "data": {
        "geometry": "spherical",
        "boundaries": [
            {"id": "full_disk", "type": "map", "files": {"Br": str(FULL_DISK_BR), "Bt": str(FULL_DISK_BT), "Bp": str(FULL_DISK_BP)}, "errors": {"Br_err": str(FULL_DISK_BR_ERR), "Bt_err": str(FULL_DISK_BT_ERR), "Bp_err": str(FULL_DISK_BP_ERR)}},
            {"id": "synoptic", "type": "map", "files": {"Br": str(SYNOPTIC_BR), "Bt": str(SYNOPTIC_BT), "Bp": str(SYNOPTIC_BP)}},
        ],
        "iterations": 10000,
        "samplers": [{"id": "random", "type": "random_radial_grouped", "batch_size": 16384, "n_lat_lon_sample": 64}],
        "validation": [{"id": "sphere", "type": "sphere"}],
        "max_radius": 1.3,
    },
    "callbacks": [{"type": "metrics", "dataset": "sphere"}],
}

if RUN_TRAINING:
    nf2.run(**config)
    wandb.finish()
else:
    config

## 5. Export And Evaluate

Export the trained state and compute the basic volume metrics used to judge whether the run completed cleanly.

In [ ]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
nf2.export_file(str(NF2_PATH), str(EXPORT_DIR / "field.vtk"), fmt="vtk", metrics=["j", "alpha", "free_energy"])

out = nf2.load(NF2_PATH)
field = out.load_spherical(radius_range=np.array(RADIUS_RANGE_SOLRAD) * u.solRad, sampling=SPHERICAL_SAMPLING, metrics=["j", "j_vec", "free_energy"], progress=True)

def values(x):
    return np.asarray(getattr(x, "value", x))

b = values(field["b"])
j = values(field["metrics"]["j"])
j_vec = values(field["metrics"]["j_vec"])
free_energy = values(field["metrics"]["free_energy"])
metrics = {
    "mean_normalized_divergence": float(np.nanmean(normalized_divergence(b))),
    "theta_J_deg": float(theta_J(b, j_vec)),
    "sigma_J_1e2": float(sigma_J(b, j_vec) * 1e2),
    "total_free_energy_density": float(np.nansum(free_energy)),
}
metrics

## 6. Quick-Look Plots

Compare the boundary map, model output, and selected radial slices for a fast visual check of the extrapolation.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
current_map = np.nansum(j, axis=0)
free_energy_map = np.nansum(free_energy, axis=0)
surface_br = b[0, :, :, 0]

for ax, image, title, cmap in [
    (axs[0], current_map, "Integrated |J|", "magma"),
    (axs[1], free_energy_map, "Free energy", "inferno"),
    (axs[2], surface_br, "Model surface Br", "RdBu_r"),
]:
    kwargs = {"origin": "lower", "aspect": "auto", "cmap": cmap}
    if cmap == "RdBu_r":
        lim = np.nanpercentile(np.abs(image), 99)
        kwargs.update(vmin=-lim, vmax=lim)
    im = ax.imshow(image, **kwargs)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    plt.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout()
plt.show()